# SoC_Driving_v13 — 프레임레이트 최적화 + 레이싱 라인

`SoC_Driving_Final (1).ipynb` 에서 두 가지를 바꾼 버전입니다.

## 1. 프레임레이트 (v13-A)

`run_20260727_120327` 실측: 프레임 주기 **198ms(5.1Hz)** 중 DPU 추론은 **12ms(6%)** 뿐이고
나머지 186ms가 캡처와 BEV 워프였습니다. 조향 목표가 200ms에 한 번만 갱신되니
그 사이는 사실상 개루프 주행이고, 코너 진입에 한 프레임 늦게 반응합니다.

- `BEV_MODE` — `legacy`(기존과 픽셀 동일) / `fast`(워프 1회로 합성) / `fastest`(리사이즈까지 제거)
- `CAPTURE_THREAD` — 캡처 전용 스레드가 항상 최신 프레임만 유지 (드라이버 버퍼 대기 제거)
- `TIMING_ENABLE` — 단계별 소요시간을 하트비트에 출력하고 텔레메트리 `stage` 필드에 기록

## 2. 레이싱 라인 (v13-B)

**누적 방위각 psi** 를 트랙 위치 인덱스로 쓰고, 구획별로 `REF_X` 를 스케줄링합니다.
코너의 총 방향전환량은 라인에 무관한 불변량이라, 라인을 튜닝해도 인덱스가 틀어지지 않습니다.

**승부처는 T4** 입니다. 랩에서 유일한 우코너인데 고정 `REF_X=198` 은 차량을 차로 중심보다
16px 왼쪽에 두므로, T4에서만 **바깥쪽**에 서 있습니다. 그래서 인쪽으로 쓸 수 있는 거리가
좌코너의 4배가 넘고(약 36px), 반경 여유도 2.44배로 가장 큽니다.

| 코너 | 회전각 | 가용 이동량 | 이득 지수 |
|---|---|---|---|
| **T4** (우) | 0.545 rad | ~36 px | **19.9** |
| T5 (좌) | 1.99 rad | ~4.5 px | 8.9 |
| T2 (좌) | 1.46 rad | ~4.5 px | 6.6 |
| T1 (좌) | 1.41 rad | ~4.5 px | 6.3 |
| T3 (좌) | 1.87 rad | 0 (반경여유 1.12x) | 0 |

T4는 또 랩에서 유일하게 `mapped` 가 양수로 유지되는 구간이라, 적분 드리프트를 리셋하는
**랜드마크** 역할도 합니다. 그것도 최대 구획 T5(랩의 22%) 직전에.

## 안전

- `RACING_LINE_ENABLE=False` 로 두면 기존 주행과 동일 → **A/B 기준선**
- `RACING_LINE_GAIN` 은 0.5에서 시작해 차선을 안 밟는 선까지 올릴 것
- 차선을 놓치면(`lost > LOST_HOLD_FRAMES`) `REF_X` 는 기본값으로 자동 복귀
- `REFX_SLEW_PX_S` 로 `REF_X` 변화율 제한 → 계단 입력이 PD 루프를 때리지 않게


## Initial Settings

In [1]:
print("Starting init...")

%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import time
import os
import sys
import math
import random
import numpy as np
import cv2
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from pynq import Overlay, MMIO, PL, allocate
from pynq.lib.video import *
from pynq_dpu import DpuOverlay
import spidev
import colorsys

import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

overlay = DpuOverlay("/home/xilinx/jupyter_notebooks/KGIC/driveCode/dpu/dpu.bit")
overlay.load_model("/home/xilinx/jupyter_notebooks/KGIC/modelCode/tiny-yolov3_256.xmodel")

print("Import & Model init completed.")

Starting init...


Import & Model init completed.


## 모터 초기설정

In [2]:
# 모터 설정
motor_0_address = 0x00A0000000
motor_1_address = 0x00A0010000
motor_2_address = 0x00A0020000
motor_3_address = 0x00A0030000
motor_4_address = 0x00A0040000
motor_5_address = 0x00A0050000

address_range = 0x10000

motor_0 = MMIO(motor_0_address , address_range)
motor_1 = MMIO(motor_1_address , address_range) # CHANGE
motor_2 = MMIO(motor_2_address , address_range)
motor_3 = MMIO(motor_3_address , address_range)
motor_4 = MMIO(motor_4_address , address_range) # CHANGE
motor_5 = MMIO(motor_5_address , address_range)

In [3]:
# 모터 기본 설정
# 300 Mhz = 3.33 ns (비바도에서 설정한 클럭 주기에 따라 달라질 수 있음. 확인 필요.)
# size = 2ms = 3.33ns x 600600.600.....
# about 600600
size = 600600
motor_0_percent = 50/100
motor_1_percent = 50/100
motor_2_percent = 50/100
motor_3_percent = 50/100
motor_4_percent = 50/100
motor_5_percent = 50/100

motor_0_duty = size * motor_0_percent
motor_1_duty = size * motor_1_percent
motor_2_duty = size * motor_2_percent
motor_3_duty = size * motor_3_percent
motor_4_duty = size * motor_4_percent
motor_5_duty = size * motor_5_percent

print("motor_0_duty:", int(motor_0_duty)) # 300300 -> 50%

motor_0.write(0x00,size)           # size
motor_0.write(0x04,int(motor_0_duty))   # DUTY
motor_0.write(0x08,0)              # valid

motor_1.write(0x00,size)  # size
motor_1.write(0x04,int(motor_1_duty))   # DUTY
motor_1.write(0x08,0)       # valid

motor_2.write(0x00,size)  # size
motor_2.write(0x04,int(motor_2_duty))   # DUTY
motor_2.write(0x08,0)       # valid

motor_3.write(0x00,size)  # size
motor_3.write(0x04,int(motor_3_duty))   # DUTY
motor_3.write(0x08,0)       # valid

motor_4.write(0x00,size)  # size
motor_4.write(0x04,int(motor_4_duty))   # DUTY
motor_4.write(0x08,0)       # valid

motor_5.write(0x00,size)  # size
motor_5.write(0x04,int(motor_5_duty))   # DUTY
motor_5.write(0x08,0)       # valid

print("Motor init completed.")

motor_0_duty: 300300
Motor init completed.


## YOLOv3 Utility functions


In [4]:
anchor_list = [10, 14, 23, 27, 37, 58, 81, 82, 135, 169, 344, 319]
anchors = np.array(anchor_list).reshape(-1, 2)

# Reshaped anchors:
# [ [10, 14,  23,  27,  37,  58],
#   [81, 82, 135, 169, 344, 319] ]

# 클래스 정보 가져오기 함수
def get_class(classes_path):
    with open(classes_path) as f:
        class_names = f.readlines()
    class_names = [c.strip() for c in class_names]
    return class_names

# lane_class.txt
# 1

classes_path = "/home/xilinx/jupyter_notebooks/KGIC/modelCode/configs/lane_class.txt"
class_names = get_class(classes_path)

num_classes = len(class_names) # 1
hsv_tuples = [(1.0 * x / num_classes , 1. , 1.) for x in range(num_classes)] # [ (1.0, 1.0, 1.0) ]
colors = list(map(lambda x: colorsys.hsv_to_rgb(*x), hsv_tuples))
colors = list(map(lambda x:
                  (int(x[0] * 255), int(x[1] * 255), int(x[2] * 255)),
                  colors)) # denormalize
random.seed(42)
random.shuffle(colors)
random.seed(None)

# 이미지 비율을 유지하며 padding하여 리사이즈하는 함수
def letterbox_image(image, size):
    ih, iw, _ = image.shape
    w, h = size
    scale = min(w/iw, h/ih)
    nw = int(iw*scale)
    nh = int(ih*scale)
    image = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)
    new_image = np.ones((h, w, 3), np.uint8) * 128
    h_start = (h-nh)//2
    w_start = (w-nw)//2
    new_image[h_start:h_start+nh, w_start:w_start+nw, :] = image
    return new_image

# 이미지 전처리 함수
def pre_process(image, model_image_size):
    image = image[..., ::-1]
    image_h, image_w, _ = image.shape
    if model_image_size != (None, None):
        assert model_image_size[0] % 32 == 0, 'Multiples of 32 required'
        assert model_image_size[1] % 32 == 0, 'Multiples of 32 required'
        boxed_image = letterbox_image(image, tuple(reversed(model_image_size)))
    else:
        new_image_size = (image_w - (image_w % 32), image_h - (image_h % 32))
        boxed_image = letterbox_image(image, new_image_size)
    image_data = np.array(boxed_image, dtype='float32')
    image_data /= 255.
    image_data = np.expand_dims(image_data, 0) 	
    return image_data

def _get_feats(feats, anchors, num_classes, input_shape):
    num_anchors = len(anchors)
    anchors_tensor = np.reshape(np.array(anchors, dtype=np.float32), [1, 1, 1, num_anchors, 2])
    grid_size = np.shape(feats)[1:3]
    nu = num_classes + 5
    predictions = np.reshape(feats, [-1, grid_size[0], grid_size[1], num_anchors, nu])
    grid_y = np.tile(np.reshape(np.arange(grid_size[0]), [-1, 1, 1, 1]), [1, grid_size[1], 1, 1])
    grid_x = np.tile(np.reshape(np.arange(grid_size[1]), [1, -1, 1, 1]), [grid_size[0], 1, 1, 1])
    grid = np.concatenate([grid_x, grid_y], axis=-1)
    grid = np.array(grid, dtype=np.float32)

    # Confidence score 계산 확인
    box_xy = (1 / (1 + np.exp(-predictions[..., :2])) + grid) / np.array(grid_size[::-1], dtype=np.float32)
    box_wh = np.exp(predictions[..., 2:4]) * anchors_tensor / np.array(input_shape[::-1], dtype=np.float32)
    box_confidence = 1 / (1 + np.exp(-predictions[..., 4:5]))
    box_class_probs = 1 / (1 + np.exp(-predictions[..., 5:]))

    return box_xy, box_wh, box_confidence, box_class_probs

def correct_boxes(box_xy, box_wh, input_shape, image_shape):
    box_yx = box_xy[..., ::-1]
    box_hw = box_wh[..., ::-1]
    input_shape = np.array(input_shape, dtype = np.float32)
    image_shape = np.array(image_shape, dtype = np.float32)
    new_shape = np.around(image_shape * np.min(input_shape / image_shape))
    offset = (input_shape - new_shape) / 2. / input_shape
    scale = input_shape / new_shape
    box_yx = (box_yx - offset) * scale
    box_hw *= scale

    box_mins = box_yx - (box_hw / 2.)
    box_maxes = box_yx + (box_hw / 2.)
    boxes = np.concatenate([
        box_mins[..., 0:1],
        box_mins[..., 1:2],
        box_maxes[..., 0:1],
        box_maxes[..., 1:2]
    ], axis = -1)
    boxes *= np.concatenate([image_shape, image_shape], axis = -1)
    return boxes


def boxes_and_scores(feats, anchors, classes_num, input_shape, image_shape):
    box_xy, box_wh, box_confidence, box_class_probs = _get_feats(feats, anchors, classes_num, input_shape)
    boxes = correct_boxes(box_xy, box_wh, input_shape, image_shape)
    boxes = np.reshape(boxes, [-1, 4])
    box_scores = box_confidence * box_class_probs
    box_scores = np.reshape(box_scores, [-1, classes_num])
    return boxes, box_scores

def nms_boxes(boxes, scores):
    """Suppress non-maximal boxes.

    # Arguments
        boxes: ndarray, boxes of objects.
        scores: ndarray, scores of objects.

    # Returns
        keep: ndarray, index of effective boxes.
    """
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2-x1+1)*(y2-y1+1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w1 = np.maximum(0.0, xx2 - xx1 + 1)
        h1 = np.maximum(0.0, yy2 - yy1 + 1)
        inter = w1 * h1

        ovr = inter / (areas[i] + areas[order[1:]] - inter)
        inds = np.where(ovr <= 0.1)[0]  # threshold
        order = order[inds + 1]

    return keep

# 모델 평가 함수: YOLOv3-Tiny의 2-출력 레이어 구조에 맞게 anchor_mask 설정
def evaluate(yolo_outputs, image_shape, class_names, anchors, threshold, max_boxes=20):
    score_thresh = threshold
    anchor_mask = [[3, 4, 5], [0, 1, 2]]  # YOLOv3-Tiny에 맞춘 2개의 출력 레이어용 anchor mask
    boxes = []
    box_scores = []
    input_shape = np.shape(yolo_outputs[0])[1:3] * np.array([32, 32])

    # anchors:
    # [ [10, 14,  23,  27,  37,  58],
    #   [81, 82, 135, 169, 344, 319] ]

    # anchors with anchor_mask ?
    # yolo_outputs == 1: 

    for i in range(len(yolo_outputs)):
        _boxes, _box_scores = boxes_and_scores(
            yolo_outputs[i], anchors[anchor_mask[i]], len(class_names),
            input_shape, image_shape)
        boxes.append(_boxes)
        box_scores.append(_box_scores)
    boxes = np.concatenate(boxes, axis=0)
    box_scores = np.concatenate(box_scores, axis=0)


    mask = box_scores >= score_thresh
    boxes_ = []
    scores_ = []
    classes_ = []
    for c in range(len(class_names)):
        class_boxes_np = boxes[mask[:, c]]
        class_box_scores_np = box_scores[:, c][mask[:, c]]

        # Non-Max Suppression with max_boxes limit
        nms_index_np = nms_boxes(class_boxes_np, class_box_scores_np)
        nms_index_np = nms_index_np[:max_boxes]  # Limit to max_boxes

        class_boxes_np = class_boxes_np[nms_index_np]
        class_box_scores_np = class_box_scores_np[nms_index_np]
        classes_np = np.ones_like(class_box_scores_np, dtype=np.int32) * c
        boxes_.append(class_boxes_np)
        scores_.append(class_box_scores_np)
        classes_.append(classes_np)

    boxes_ = np.concatenate(boxes_, axis=0)
    scores_ = np.concatenate(scores_, axis=0)
    classes_ = np.concatenate(classes_, axis=0)


    return boxes_, scores_, classes_


## Flight Recorder (블랙박스)

In [5]:
# ===== 플라이트 레코더 (블랙박스): 주행 데이터를 보드 로컬 디스크에 기록 =====
# 목적: 주행 문제가 인식(YOLO)/판단(각도 변환)/제어(폐루프) 중 어느 레이어인지 사후 분석.
# 설계: 주행 루프는 큐에 넣기만 하고(논블로킹), JPEG 인코딩/디스크 쓰기는 별도 쓰레드가 수행
#       -> 기록이 비전/제어 주기를 방해하지 않음. 큐가 가득 차면 해당 프레임은 버림(주행 우선).
# 분석: 주행 후 run_* 폴더를 맥으로 복사해서 replay_viewer.html 로 열기.
#   scp -r xilinx@<보드IP>:/home/xilinx/jupyter_notebooks/KGIC/driveCode/logs/run_* ~/AIchip/logs/
import json
import queue
import threading
from datetime import datetime

RECORD_BASE_DIR = "/home/xilinx/jupyter_notebooks/KGIC/driveCode/logs"
RECORD_JPEG_QUALITY = 70   # 낮출수록 용량 절약. 70이면 프레임당 대략 20~40KB

class FlightRecorder:
    def __init__(self, base_dir=RECORD_BASE_DIR, jpeg_quality=RECORD_JPEG_QUALITY):
        self.run_dir = os.path.join(base_dir, datetime.now().strftime("run_%Y%m%d_%H%M%S"))
        os.makedirs(os.path.join(self.run_dir, "frames"), exist_ok=True)
        self.jpeg_quality = jpeg_quality
        self.q = queue.Queue(maxsize=120)   # 약 수 초 분량 버퍼
        self.dropped = 0
        self._stop = False
        self._f = open(os.path.join(self.run_dir, "telemetry.jsonl"), "w")
        self._writer = threading.Thread(target=self._write_loop, daemon=True)
        self._writer.start()
        print(f"[REC] 기록 시작: {self.run_dir}")

    def record(self, vis_img, telem):
        # 주행 루프에서 매 프레임 호출. 절대 블로킹하지 않는다.
        try:
            # vis_img는 주행 루프가 다음 프레임에서 덮어쓸 수 있는 버퍼라 복사해서 넘김
            img = None if vis_img is None else vis_img.copy()
            self.q.put_nowait((img, telem))
        except queue.Full:
            self.dropped += 1

    def _write_loop(self):
        while not (self._stop and self.q.empty()):
            try:
                img, telem = self.q.get(timeout=0.2)
            except queue.Empty:
                continue
            if img is not None:
                fname = f"frames/{telem['i']:06d}.jpg"
                cv2.imwrite(os.path.join(self.run_dir, fname), img,
                            [cv2.IMWRITE_JPEG_QUALITY, self.jpeg_quality])
                telem["frame_file"] = fname
            self._f.write(json.dumps(telem) + "\n")

    def close(self):
        self._stop = True
        self._writer.join(timeout=10.0)
        self._f.flush()
        self._f.close()
        print(f"[REC] 기록 종료: {self.run_dir} (드롭된 프레임 {self.dropped}개)")

print("FlightRecorder ready")

FlightRecorder ready


## DPU Setting

In [6]:
# DPU: Deep Learning Processing Unit

# 1. .xmodel 파일에서 실행기(runner) 객체를 가져오기. 여기서 실제 추론이 일어남.
dpu = overlay.runner

# 2. 모델에 사전 정의된 입력/출력 텐서의 구조와 타입을 가져옴.
# tiny_yolov는 입력 1개, 출력 2개(레이어) 구조.
inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

# 3. 가져온 텐서의 shape을 출력.
# 13 * 13은 큰 객체 (coarse grid), 26 * 26은 작은 객체 (fine grid) 검출용.
# 배치란? 한 번에 이미지 몇 장을 처리할 것이냐는 뜻. 
# 자율주행은 이미지 한 장씩 처리하지만 병렬 처리의 효율화를 위해 1 이상의 배치를 사용하는 경우가 많음.
shapeIn = tuple(inputTensors[0].dims) # (1, 256, 256, 3) -> (배치, 높이, 폭, 채널(RGB))

if shapeIn != (1, 256, 256, 3):
    logger.warning("inputTensors의 shape이 예상과 다름. 예상: (1, 256, 256, 3), 실제: %s", shapeIn)

shapeOut0 = (tuple(outputTensors[0].dims)) # (1, 13, 13, 75)
shapeOut1 = (tuple(outputTensors[1].dims)) # (1, 26, 26, 75)

# 4. (?) 출력 텐서 전체 크기를 배치 크기(shapeIn[0], 보통 1)로 나눠서 1개 이미지 당 출력 원소 개수 계산.
# 여기 말고는 쓰이는 곳이 없는데 지워도 될 듯? 아마도 디버깅 용.
outputSize0 = int(outputTensors[0].get_data_size() / shapeIn[0]) # 12675 (= 13 * 13 * 75)
outputSize1 = int(outputTensors[1].get_data_size() / shapeIn[0]) # 50700 (= 26 * 26 * 75)

# 5. DPU 추론 용 버퍼 할당.
# np.empty라 초기화가 없어 소폭 빠르다. C-style buffer (행 우선 방식, 버퍼에 행 단위로 직렬화되어있다는 뜻)
input_data = [np.empty(shapeIn, dtype=np.float32, order="C")]
output_data = [np.empty(shapeOut0, dtype=np.float32, order="C"),
               np.empty(shapeOut1, dtype=np.float32, order="C")]

# image는 input_data[0]의 단순 alias. 같은 주소를 가리킴.
image = input_data[0]

## Driving Funtions

In [ ]:
import threading

# ================== 튜닝 파라미터 (트랙에서 조정) ==================
DRIVE_SPEED = 100          # 구동 속도(0~100)
STEER_KP = 0.026          # 조향 P 게인: 오차 1도당 duty 증가량
STEER_KD = 0.035          # 조향 D 감쇠: 바퀴가 목표 방향으로 움직일 때 duty를 줄이는 양
STEER_MIN_DUTY = 0.52     # 조향 모터가 실제로 움직이기 시작하는 최소 duty
STEER_MAX_DUTY = 0.90     # 조향 모터 최대 duty 제한
STEER_DEADZONE = 1.2      # 이 오차(도) 이내면 목표 도달로 보고 조향 모터 정지
STEER_D_FILTER = 0.35     # mapped 변화량 저역통과. 클수록 D가 민감해짐
LOST_HOLD_FRAMES = 5      # 차선 미검출 직후 이 프레임까지는 마지막 조향 유지
STEER_TRIM = 0            # 조향 중립 트림(실측): 물리적 직진의 mapped값.
REF_X_BASE = 198          # 조향 기준 x값의 기본값 (직선 정렬상태에서 우측차선 중심 x좌표)
REF_X = REF_X_BASE        # 하위호환 별칭. 실제 주행 중에는 RobotController.ref_x가 쓰임.
STEER_DIR = +1            # 스티어링 부호. 핸들이 거꾸로 움직이면 음수로 바꾸기.
STEER_GAIN = 60.0 / 128.0 # 화면 오프셋(±128px) -> 조향 타겟(±20)
CENTER_EMA = 1            # 차선중심 저역통과(0~1, 클수록 민감). 조향 지터 완화
CONTROL_HZ = 100          # 제어 스레드 주기(Hz)
DISPLAY_EVERY = 1         # N프레임마다 1번만 화면 표시
YOLO_THRESHOLD = 0.01     # YOLO 임계값
USE_COLOR_FILTER = False  # True: 채도 기반 차선 색상 필터 사용
LANE_MAX_SATURATION = 40  # 차선 색상 필터 임계값(HSV 채도 0~255)

# ---- 화각(FOV) 관련 ----
CAPTURE_W = 1920          # 카메라 캡처 폭(px)
CAPTURE_H = 1080          # 카메라 캡처 높이(px)
BEV_WORK_HEIGHT = 480     # BEV 계산용 작업 높이
SRC_RATIO = [(238/640, 316/480), (402/640, 313/480), (501/640, 476/480), (155/640, 476/480)]
CUTTING_RATIO = 300/480   # ROI 시작 비율

# =====================================================================
# [v13-A] 프레임레이트 최적화 스위치
# =====================================================================
# run_20260727_120327 실측: 프레임 주기 198ms(5.1Hz) 중 DPU 추론은 12ms(6%)뿐.
# 나머지 186ms가 캡처 + BEV 워프. 아래 스위치로 단계별 개선을 A/B 테스트한다.
#
#   BEV_MODE = "legacy"  : 원본 -> work(853x480) 리사이즈 -> 전체 워프 -> 하단 crop -> 256 리사이즈
#                          (기존 코드와 픽셀 단위로 동일. 기준선 측정용)
#   BEV_MODE = "fast"    : 원본 -> work 리사이즈 -> [work->256x256 단일 워프]
#                          전체 크기 워프와 crop/resize를 한 번의 워프로 합성. 화질 동일, 대폭 단축.
#   BEV_MODE = "fastest" : 원본 -> [원본->256x256 단일 워프]  (리사이즈까지 제거)
#                          가장 빠르지만 1920->256 직접 샘플링이라 에일리어싱 위험. 검출률 확인 필수.
BEV_MODE = "fastest"      # 보드 실측: legacy 29.4 / fast 18.8 / fastest 8.3 ms
                          # 세 모드 화상이 육안상 동일하고 화소차도 평균 2.6뿐이라 fastest 채택.
                          # 단 BEV는 198ms 중 29ms(15%)일 뿐이다 — 진짜 병목은 캡처(157ms).
                          # 검출률이 떨어지면 "fast"로 되돌릴 것.

CAPTURE_THREAD = True     # True: 캡처 전용 스레드가 항상 최신 프레임만 들고 있음(대기시간 제거)
CAPTURE_MJPG = True       # True: MJPG 포맷 요청. USB 대역폭 병목 완화(카메라가 지원할 때만 효과)
CAPTURE_BUFSIZE_1 = True  # True: 드라이버 버퍼를 1로 -> 오래된 프레임 누적 방지
TIMING_ENABLE = True      # True: 파이프라인 단계별 소요시간 측정 + 텔레메트리 기록

# ---- 블랙박스 모드 스위치 ----
RECORD_ENABLE = True      # True: FlightRecorder로 매 프레임 디스크 기록
DISPLAY_ENABLE = False    # False: 노트북 실시간 화면표시 끔
HEARTBEAT_EVERY = 30      # N프레임마다 한 줄만 상태 출력

# =====================================================================
# [v13-B] 레이싱 라인 — 구획별 REF_X 스케줄링
# =====================================================================
# 근거: run_20260727_120327 (프레임 0~233, 랩 46.35s) 를 조향 피드백 적분으로 복원.
#
# 위치 인덱스로 "누적 방위각 psi"를 쓴다. psi = ∫ PSI_K * mapped dt.
#   왜 시간/거리가 아니라 방위각인가:
#     코너의 총 방향전환량은 '트랙이 꺾인 각도'라 어떤 라인으로 돌든 불변이다.
#     반면 시간·거리 인덱스는 REF_X를 바꾸는 순간 전부 틀어져서, 라인을 튜닝할 때마다
#     인덱스를 다시 재야 한다. 즉 psi는 지금 최적화하려는 대상이 오염시키지 않는 유일한 좌표.
#     덤으로 속도 변동에도 면역이다.
#   단, psi는 코너 '안'에서만 쓴다. 아래 코너 테이블 주석 참조.
PSI_K = 0.9117            # mapped 1단위를 1초 유지할 때의 방위각 변화(도). 로그 폐합으로 교정.

RACING_LINE_ENABLE = True # False로 두면 REF_X_BASE 고정 = 기존 주행(A/B 기준선)
RACING_LINE_GAIN = 0.5    # 0~1. 라인 공격성 전역 스케일. 0.5에서 시작해 차선을 안 밟는 선까지 올릴 것.

# REF_X 부호 규약 (중요):
#   REF_X를 낮추면 -> 제어기가 우측 차선을 화면 왼쪽으로 보내려 함 -> 차량이 오른쪽으로 이동
#   REF_X를 높이면 -> 차량이 왼쪽으로 이동
# 따라서 delta 양수 = 차량이 왼쪽(좌코너의 인사이드), 음수 = 오른쪽(우코너의 인사이드).
#
# ---- 코너 테이블 ----
# psi를 '구획 경계'로 쓰지 않는 이유(실측으로 드러난 함정):
#   직선은 정의상 방향이 안 변한다. S2는 7.4초나 되는데 psi 폭이 4°뿐이고, S3는 0.9°다.
#   즉 psi로는 직선 위 어디인지 알 수 없고, 경계를 psi 임계값으로 잡으면 노이즈에 뚫린다.
#   (실제로 초기 구현은 S3를 4프레임 일찍 통과했고, T4 탈출 임계값 -245.9°에 0.9° 못 미쳐
#    T5로 영영 넘어가지 못했다.)
# 그래서 역할을 나눈다:
#   - 어느 코너인가  -> |mapped| 로 판정. 실측상 직선 최대 5.7 vs 코너 최대 16~20으로 깨끗이 갈린다.
#   - 코너 안 어디인가 -> psi. 코너 안에서는 81~114°씩 변해서 분해능이 충분하다.
# 코너 순서는 트랙이 고정이라 불변이고, T4만 유일하게 부호가 +라서 자기 위치를 스스로 증명한다.
#
#          이름   dpsi(도)  부호   d_진입  d_정점  d_탈출
TRACK_CORNERS = [
    ("T1",  -81.0,  -1,     0,     +8,     0),   # 좌. 반경여유 1.57x. 소폭 인쪽.
    ("T2",  -83.4,  -1,     0,     +6,     0),   # 좌. 여유 1.27x, 포화 2.0s라 보수적으로.
    ("T3", -107.3,  -1,    -6,     -4,   -18),   # 좌. 여유 1.12x = 인쪽 라인이 물리적으로 불가.
                                                 # 목표는 단축이 아니라 (a) 밀려나지 않게 넓게 돌고
                                                 # (b) 탈출을 오른쪽으로 빼서 T4 인사이드를 미리 잡는 것.
    ("T4",  +31.2,  +1,   -28,    -34,   -14),   # ★ 승부처. 유일한 우코너 = 인사이드가 오른쪽.
                                                 # 고정 REF_X가 방향을 틀리는 유일한 구간이라 가용 이동량 4배.
                                                 # 여유 2.44x로 가장 커서 조향 포화 걱정도 없음.
    ("T5", -114.1,  -1,    -8,     +8,     0),   # 좌. 랩의 22%. T4 탈출(오른쪽)에서 인쪽으로 감아 들어감.
]

CORNER_ENTER = 8.0        # |mapped|가 이 값을 넘고 유지되면 코너 진입 (직선 최대 실측 5.7)
CORNER_EXIT = 5.0         # 이 값 아래로 떨어지고 유지되면 코너 탈출 (히스테리시스)
CORNER_HOLD_SEC = 0.25    # 진입/탈출 판정 유지시간. 순간 노이즈로 상태가 튀는 것 방지.

REFX_DELTA_LIMIT = 40.0   # |delta| 상한(px). 차선 침범 방지용 하드 클램프.
REFX_SLEW_PX_S = 25.0     # REF_X 변화율 상한(px/s). 계단 입력이 PD루프를 때리는 것을 막는다.
                          # 낮추면 안전하지만 코너 안에서 목표 위치에 못 도달할 수 있음.
# =====================================================================

# SPI 초기화 (조향 가변저항 피드백)
spi0 = spidev.SpiDev()
spi0.open(0, 0)
spi0.max_speed_hz = 20000000
spi0.mode = 0b00


class StageTimer:
    """파이프라인 단계별 소요시간 측정. with 블록으로 감싸면 누적된다."""

    def __init__(self, enable=True):
        self.enable = enable
        self.last = {}      # 이번 프레임 각 단계 ms
        self._acc = {}      # 하트비트 구간 누적
        self._n = 0

    class _Scope:
        def __init__(self, owner, name):
            self.owner = owner
            self.name = name

        def __enter__(self):
            self.t0 = time.time()
            return self

        def __exit__(self, *exc):
            ms = (time.time() - self.t0) * 1000.0
            self.owner.last[self.name] = round(ms, 2)
            self.owner._acc[self.name] = self.owner._acc.get(self.name, 0.0) + ms
            return False

    def stage(self, name):
        if not self.enable:
            return _NullScope()
        return StageTimer._Scope(self, name)

    def frame_done(self):
        self._n += 1

    def summary(self):
        """하트비트용 한 줄 요약. 호출하면 누적치를 리셋한다."""
        if not self.enable or self._n == 0:
            return ""
        parts = ["%s %.1f" % (k, v / self._n) for k, v in sorted(self._acc.items(), key=lambda kv: -kv[1])]
        total = sum(self._acc.values()) / self._n
        self._acc = {}
        self._n = 0
        return "[%.1fms 총] " % total + " | ".join(parts)


class _NullScope:
    def __enter__(self):
        return self

    def __exit__(self, *exc):
        return False


class FrameGrabber:
    """캡처 전용 스레드. 항상 '가장 최신' 프레임 한 장만 들고 있는다.

    cap.read()를 주행 루프에서 직접 부르면 드라이버 버퍼에 쌓인 오래된 프레임을
    순서대로 받게 되고, 그만큼 조향이 과거를 보고 반응한다(= 코너 진입 지연).
    여기서는 grab을 계속 돌려 버려서 주행 루프가 늘 최신 한 장만 가져가게 한다.
    """

    def __init__(self, cap):
        self.cap = cap
        self.lock = threading.Lock()
        self.frame = None
        self.seq = 0
        self.dropped = 0
        self.stop_flag = False
        self.t = threading.Thread(target=self._loop, daemon=True)
        self.t.start()

    def _loop(self):
        while not self.stop_flag:
            ok, f = self.cap.read()
            if not ok:
                time.sleep(0.005)
                continue
            with self.lock:
                if self.frame is not None:
                    self.dropped += 1   # 주행 루프가 못 가져간 프레임 = 버려짐
                self.frame = f
                self.seq += 1

    def read_latest(self, timeout=1.0):
        t0 = time.time()
        while time.time() - t0 < timeout:
            with self.lock:
                if self.frame is not None:
                    f = self.frame
                    self.frame = None
                    return True, f
            time.sleep(0.001)
        return False, None

    def stop(self):
        self.stop_flag = True
        self.t.join(timeout=1.0)


class TrackModel:
    """복원한 트랙을 내장하고, 현재 위치에 맞는 REF_X를 돌려준다.

    - 코너 판정: |mapped| (직선과 코너가 조향 크기로 깨끗이 갈린다)
    - 코너 내 진행도: psi 누적 방위각 (코너 안에서만 분해능이 충분하다)
    - T4는 랩에서 유일한 우코너라, 부호만으로 자기 위치를 증명한다 -> 하드 재동기화 지점
    """

    def __init__(self):
        self.psi = 0.0
        self.corner_idx = -1        # 아직 첫 코너 전
        self.in_corner = False
        self.psi_entry = 0.0
        self.lap_done = False
        self._enter_hold = 0.0
        self._exit_hold = 0.0
        self._t4_hold = 0.0
        self.ref_x = float(REF_X_BASE)
        self.last_delta = 0.0
        self.resync_count = 0

    def _t4_index(self):
        for i, c in enumerate(TRACK_CORNERS):
            if c[2] > 0:
                return i
        return -1

    # ---- 100Hz 제어 스레드에서 호출 ----
    def update(self, mapped, dt):
        if dt <= 0 or dt > 0.5:
            return
        self.psi += PSI_K * mapped * dt
        am = abs(mapped)

        # --- T4 하드 재동기화: 랩에서 우조향이 유지되는 곳은 T4뿐 ---
        t4 = self._t4_index()
        if t4 >= 0 and mapped > CORNER_ENTER:
            self._t4_hold += dt
            if self._t4_hold >= CORNER_HOLD_SEC and self.corner_idx != t4:
                # 정상 흐름에서도 여기가 한 번 걸린다: 우조향 감지가 T3 탈출 판정보다 먼저 끝나면
                # 순차 카운터(T3->T4)를 앞질러 들어온다. 그건 이상이 아니라 랜드마크가 일을 한 것.
                # 진짜 이상은 '직전 코너가 T3가 아니었을 때' — 그때만 카운트해서 로그로 남긴다.
                if self.corner_idx not in (t4 - 1, t4):
                    self.resync_count += 1
                self.corner_idx = t4
                self.in_corner = True
                self.psi_entry = self.psi
        else:
            self._t4_hold = 0.0

        if not self.in_corner:
            # 코너 진입 감지
            if am > CORNER_ENTER:
                self._enter_hold += dt
                if self._enter_hold >= CORNER_HOLD_SEC:
                    nxt = self.corner_idx + 1
                    if nxt < len(TRACK_CORNERS):
                        self.corner_idx = nxt
                        self.in_corner = True
                        self.psi_entry = self.psi
                    self._enter_hold = 0.0
                    self._exit_hold = 0.0
            else:
                self._enter_hold = 0.0
        else:
            # 코너 탈출 감지
            if am < CORNER_EXIT:
                self._exit_hold += dt
                if self._exit_hold >= CORNER_HOLD_SEC:
                    self.in_corner = False
                    self._exit_hold = 0.0
                    self._enter_hold = 0.0
                    if self.corner_idx >= len(TRACK_CORNERS) - 1:
                        self.lap_done = True
            else:
                self._exit_hold = 0.0

    def progress(self):
        """현재 코너 안에서의 진행도 0~1. 코너 밖이면 None."""
        if not self.in_corner or not (0 <= self.corner_idx < len(TRACK_CORNERS)):
            return None
        dpsi = TRACK_CORNERS[self.corner_idx][1]
        if dpsi == 0:
            return 0.0
        return float(min(1.0, max(0.0, (self.psi - self.psi_entry) / dpsi)))

    # ---- 비전 루프에서 호출: REF_X (슬루 제한 적용) ----
    def compute_ref_x(self, dt, lane_ok=True):
        if not RACING_LINE_ENABLE or not lane_ok:
            target_delta = 0.0
        elif self.in_corner and 0 <= self.corner_idx < len(TRACK_CORNERS):
            _, _, _, d_in, d_apex, d_out = TRACK_CORNERS[self.corner_idx]
            p = self.progress()
            target_delta = float(np.interp(p, [0.0, 0.5, 1.0], [d_in, d_apex, d_out]))
        else:
            # 코너 사이 직선: 다음 코너의 진입 위치를 미리 잡아둔다(사전 배치).
            nxt = self.corner_idx + 1
            target_delta = TRACK_CORNERS[nxt][3] if nxt < len(TRACK_CORNERS) else 0.0

        target_delta *= RACING_LINE_GAIN
        target_delta = float(np.clip(target_delta, -REFX_DELTA_LIMIT, REFX_DELTA_LIMIT))
        self.last_delta = target_delta
        target_ref = REF_X_BASE + target_delta

        # 슬루 제한: REF_X가 계단으로 바뀌면 (center - REF_X)가 튀어 PD 루프가 풀락으로 때린다.
        max_step = REFX_SLEW_PX_S * max(0.0, min(dt, 0.5))
        diff = target_ref - self.ref_x
        if abs(diff) > max_step:
            diff = max_step if diff > 0 else -max_step
        self.ref_x += diff
        return self.ref_x

    def seg_name(self):
        if self.in_corner and 0 <= self.corner_idx < len(TRACK_CORNERS):
            return TRACK_CORNERS[self.corner_idx][0]
        nxt = self.corner_idx + 1
        if nxt < len(TRACK_CORNERS):
            return "S>" + TRACK_CORNERS[nxt][0]      # 다음 코너를 향한 직선
        return "S>FIN"


class RobotController:
    # 구조: 제어 폐루프는 빠른 별도 스레드(100Hz), 카메라/DPU 루프는 목표 조향각만 갱신.
    def __init__(self):
        self.image_processor = ImageProcessor()
        self.size = 600600  # clock count for 2ms in 300 Mhz clock

        self.left_speed = 0
        self.right_speed = 0
        self.steering_angle = 0.0
        self.speed = DRIVE_SPEED

        # PD 제어 상태
        self._prev_mapped_resistance = None
        self._steer_velocity_ema = 0.0

        # 가변저항 실측값
        self.resistance_most_left = 1883
        self.resistance_most_right = 1294

        self.stop_flag = False
        self._center_ema = None
        self._lost_frames = 0
        self._last_seen_steering_angle = 0.0
        self._dbg_target = 0.0
        self._dbg_mapped = 0.0
        self._dbg_adc = 0
        self._dbg_cmd = "stay"
        self._dbg_duty = 0.0
        self.recorder = None

        # 레이싱 라인
        self.track = TrackModel()
        self.ref_x = float(REF_X_BASE)
        self._ctrl_prev_t = None

        # 계측
        self.timer = StageTimer(TIMING_ENABLE)
        self.grabber = None
        # 차선 밟음 라벨링 (앞 셀에서 만든 wheel_labeler를 있으면 자동으로 붙인다)
        self.labeler = globals().get("wheel_labeler") if globals().get("WHEEL_LABEL_ENABLE", False) else None

        self.init_motors()

    def init_motors(self):
        for motor in [motor_0, motor_1, motor_2, motor_3, motor_4, motor_5]:
            motor.write(0x00, self.size)
            motor.write(0x04, 0)
            motor.write(0x08, 0)

    def map_value(self, x, in_min, in_max, out_min, out_max):
        if x <= in_min:
            return out_max
        elif x >= in_max:
            return out_min
        else:
            return (in_max - x) * (out_max - out_min) / (in_max - in_min) + out_min

    def read_adc(self, spi):
        r = spi.xfer2([0x00, 0x00])
        return ((r[0] & 0x0F) << 8) | r[1]

    # ---- 조향 ----
    def right(self, duty_ratio):
        duty_val = int(self.size * float(np.clip(duty_ratio, 0.0, 1.0)))
        motor_4.write(0x08, 0)
        motor_5.write(0x04, duty_val)
        motor_5.write(0x08, 1)

    def left(self, duty_ratio):
        duty_val = int(self.size * float(np.clip(duty_ratio, 0.0, 1.0)))
        motor_5.write(0x08, 0)
        motor_4.write(0x04, duty_val)
        motor_4.write(0x08, 1)

    def stay(self):
        motor_5.write(0x08, 0)
        motor_4.write(0x08, 0)
        motor_5.write(0x04, 0)
        motor_4.write(0x04, 0)

    def pd_steer_duty(self, error, mapped_resistance):
        abs_error = abs(error)

        if self._prev_mapped_resistance is None:
            raw_velocity = 0.0
        else:
            raw_velocity = mapped_resistance - self._prev_mapped_resistance
        self._prev_mapped_resistance = mapped_resistance

        self._steer_velocity_ema = (
            STEER_D_FILTER * raw_velocity +
            (1 - STEER_D_FILTER) * self._steer_velocity_ema
        )

        duty_ratio = STEER_MIN_DUTY + (STEER_KP * abs_error)
        if error * self._steer_velocity_ema > 0:
            duty_ratio -= STEER_KD * abs(self._steer_velocity_ema)

        return float(np.clip(duty_ratio, STEER_MIN_DUTY, STEER_MAX_DUTY))

    # ---- 구동 ----
    def set_left_speed(self, speed):
        duty = int(self.size * (abs(speed) / 100))
        motor_2.write(0x04, duty)
        motor_3.write(0x04, duty)
        if speed > 0:
            motor_3.write(0x08, 0)
            motor_2.write(0x08, 1)
        else:
            motor_3.write(0x08, 1)
            motor_2.write(0x08, 0)

    def set_right_speed(self, speed):
        duty = int(self.size * (abs(speed) / 100))
        motor_1.write(0x04, duty)
        motor_0.write(0x04, duty)
        if speed > 0:
            motor_1.write(0x08, 0)
            motor_0.write(0x08, 1)
        else:
            motor_1.write(0x08, 1)
            motor_0.write(0x08, 0)

    # ---- 조향 폐루프: PD 제어 + 방위각 적분 ----
    def control_motors(self):
        adc_raw = self.read_adc(spi0)
        mapped_resistance = self.map_value(
            adc_raw,
            self.resistance_most_right,
            self.resistance_most_left,
            -20, 20
        )

        # 트랙 위치 추정: 실제 바퀴각을 100Hz로 적분 (비전 주기와 무관)
        now = time.time()
        if self._ctrl_prev_t is not None:
            self.track.update(mapped_resistance, now - self._ctrl_prev_t)
        self._ctrl_prev_t = now

        target = self.steering_angle + STEER_TRIM
        self._dbg_target = target
        self._dbg_mapped = mapped_resistance
        self._dbg_adc = adc_raw

        error = target - mapped_resistance
        abs_error = abs(error)

        if abs_error < STEER_DEADZONE:
            self._dbg_cmd = "stay"
            self._dbg_duty = 0.0
            self.stay()
        else:
            duty_ratio = self.pd_steer_duty(error, mapped_resistance)
            self._dbg_duty = duty_ratio
            if error < 0:
                self._dbg_cmd = "left"
                self.left(duty_ratio)
            else:
                self._dbg_cmd = "right"
                self.right(duty_ratio)

        self.set_left_speed(self.left_speed)
        self.set_right_speed(self.right_speed)

    def control_loop(self):
        period = 1.0 / CONTROL_HZ
        while not self.stop_flag:
            self.control_motors()
            time.sleep(period)

    # ---- 비전 결과 -> 조향 타겟(±20) ----
    def vision_to_target(self, center):
        if center is None:
            return 0.0
        offset = center - self.ref_x
        target = STEER_DIR * offset * STEER_GAIN
        return float(np.clip(target, -20, 20))

    # ---- 메인 루프 ----
    def run(self, camera_index=0):
        cap = cv2.VideoCapture(camera_index)
        if CAPTURE_MJPG:
            try:
                cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
            except Exception as e:
                logger.warning("MJPG 설정 실패(무시하고 진행): %s", e)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAPTURE_W)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAPTURE_H)
        if CAPTURE_BUFSIZE_1:
            try:
                cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
            except Exception as e:
                logger.warning("BUFFERSIZE 설정 실패(무시하고 진행): %s", e)
        if not cap.isOpened():
            logger.critical("카메라를 열 수 없습니다.")
            return

        print("[CFG] BEV_MODE=%s  CAPTURE_THREAD=%s  MJPG=%s" % (BEV_MODE, CAPTURE_THREAD, CAPTURE_MJPG))
        print("[CFG] RACING_LINE=%s  GAIN=%.2f  SLEW=%.0fpx/s" % (RACING_LINE_ENABLE, RACING_LINE_GAIN, REFX_SLEW_PX_S))

        if CAPTURE_THREAD:
            self.grabber = FrameGrabber(cap)

        if RECORD_ENABLE:
            self.recorder = FlightRecorder()

        self.left_speed = self.speed
        self.right_speed = self.speed
        ctrl = threading.Thread(target=self.control_loop, daemon=True)
        ctrl.start()

        image_widget = widgets.Image(format='jpeg')
        frame_i = 0
        prev_loop_t = time.time()
        try:
            while not self.stop_flag:
                with self.timer.stage("capture"):
                    if self.grabber is not None:
                        ret, frame = self.grabber.read_latest()
                    else:
                        ret, frame = cap.read()
                if not ret:
                    logger.critical("프레임을 읽을 수 없습니다.")
                    break

                now = time.time()
                loop_dt = now - prev_loop_t
                prev_loop_t = now

                angle, vis = self.image_processor.process_frame(frame, timer=self.timer)
                center = self.image_processor.last_lane_center

                # 사람이 라벨링 중인 '차선을 밟은 바퀴'. 주행 제어에는 일절 쓰지 않고 기록만 한다.
                touched = self.labeler.note(self.labeler.read()) if self.labeler else []

                with self.timer.stage("control"):
                    # 레이싱 라인: 현재 구획에 맞는 REF_X (차선 놓치면 기본값으로 복귀)
                    lane_ok = (center is not None) or (self._lost_frames <= LOST_HOLD_FRAMES)
                    self.ref_x = self.track.compute_ref_x(loop_dt, lane_ok=lane_ok)
                    self.image_processor.reference_point_x = int(round(self.ref_x))

                    if center is not None:
                        self._lost_frames = 0
                        if self._center_ema is None:
                            self._center_ema = float(center)
                        else:
                            self._center_ema = CENTER_EMA * center + (1 - CENTER_EMA) * self._center_ema
                        self.steering_angle = self.vision_to_target(self._center_ema)
                        self._last_seen_steering_angle = self.steering_angle
                    else:
                        self._lost_frames += 1
                        if self._lost_frames <= LOST_HOLD_FRAMES:
                            self.steering_angle = self._last_seen_steering_angle
                        else:
                            self.steering_angle = 0.0
                            self._center_ema = None

                if self.recorder is not None:
                    ip = self.image_processor
                    telem = {
                        "t": round(time.time(), 3),
                        "i": frame_i,
                        "center": center,
                        "center_ema": None if self._center_ema is None else round(self._center_ema, 2),
                        "lost": self._lost_frames,
                        "angle": None if center is None else round(float(angle), 2),
                        "steer": round(self.steering_angle, 2),
                        "target": round(self._dbg_target, 2),
                        "mapped": round(self._dbg_mapped, 2),
                        "adc": self._dbg_adc,
                        "cmd": self._dbg_cmd,
                        "duty": round(self._dbg_duty, 3),
                        "exec_ms": round(ip.last_exec_ms, 1),
                        "boxes": ip.last_boxes,
                        "scores": ip.last_scores,
                        # --- v13 신규 ---
                        "psi": round(self.track.psi, 2),        # 누적 방위각(도) = 트랙 위치
                        "seg": self.track.seg_name(),           # 현재 구획 (코너명 또는 S>다음코너)
                        "ref_x": round(self.ref_x, 1),          # 이번 프레임 조향 기준 x
                        "refd": round(self.track.last_delta, 1),# 기본값 대비 라인 오프셋
                        "prog": self.track.progress(),          # 코너 내 진행도 0~1
                        "rsync": self.track.resync_count,       # 비정상 재동기화 횟수(0이 정상)
                        "wheel": touched,                       # [라벨] 이 시점에 차선을 밟은 바퀴
                        "loop_ms": round(loop_dt * 1000.0, 1),  # 실제 루프 주기
                        "stage": dict(self.timer.last) if TIMING_ENABLE else None,
                    }
                    self.recorder.record(vis, telem)

                self.timer.frame_done()

                if frame_i % HEARTBEAT_EVERY == 0:
                    drop = self.grabber.dropped if self.grabber else 0
                    print("[%s] i=%d psi=%+.0f ref_x=%.0f(%+.0f) center=%s "
                          "target=%+.1f mapped=%+.1f cmd=%s" % (
                              self.track.seg_name(), frame_i, self.track.psi, self.ref_x,
                              self.track.last_delta, center, self._dbg_target,
                              self._dbg_mapped, self._dbg_cmd))
                    if TIMING_ENABLE:
                        print("      %s  (drop %d)" % (self.timer.summary(), drop))
                    if self.labeler is not None:
                        print("      [LABEL] %s" % self.labeler.summary())

                frame_i += 1
                if DISPLAY_ENABLE and frame_i % DISPLAY_EVERY == 0:
                    ok, jpeg = cv2.imencode('.jpeg', vis)
                    if ok:
                        image_widget.value = jpeg.tobytes()
                        display(image_widget)
                        clear_output(wait=True)
        except KeyboardInterrupt:
            print("사용자 중지 (Ctrl+C)")
        finally:
            self.stop_flag = True
            ctrl.join(timeout=1.0)
            if self.grabber is not None:
                self.grabber.stop()
            cap.release()
            if self.recorder is not None:
                self.recorder.close()
            if self.labeler is not None:
                print("[LABEL] 최종 집계 (%d프레임): %s" % (self.labeler.frames, self.labeler.summary()))
            self.cleanup()

    def cleanup(self):
        for motor in [motor_0, motor_1, motor_2, motor_3, motor_4, motor_5]:
            motor.write(0x00, 0)
            motor.write(0x04, 0)
            motor.write(0x08, 0)
        clear_output(wait=True)
        print("모터 정지 및 종료.")


print("RobotController(v13: %s BEV / racing-line %s @gain %.2f) ready"
      % (BEV_MODE, "ON" if RACING_LINE_ENABLE else "OFF", RACING_LINE_GAIN))


## ImageProcessor

In [ ]:
class ImageProcessor:
    # 비전 파이프라인: BEV(학습 전처리와 동일) -> ROI -> 256 리사이즈 -> DPU -> 차선검출 -> 각도
    #
    # [v13] 기존 파이프라인은 프레임당 186ms를 쓰고 있었다(DPU는 12ms뿐).
    #   원본1920x1080 --resize--> work 853x480 --warp(853x480 전체)--> BEV
    #        --crop(하단 180행)--> --resize--> 256x256
    #   여기서 853x480 전체를 워프한 뒤 대부분을 버린다. 워프 비용은 '출력 크기'에 비례하므로
    #   crop/resize까지 하나의 호모그래피로 합성해 256x256로 직접 워프하면 낭비가 사라진다.
    #   변환 행렬은 프레임 크기가 같으면 매번 동일하므로 캐싱한다(compute_bev_geometry 매프레임 재계산 제거).

    def __init__(self):
        self.point_detection_height = 20   # 검출점 y (256 리사이즈 기준)
        self.reference_point_x = REF_X_BASE  # 각도/조향 기준점 x. 주행 중 RobotController가 매 프레임 갱신.
        self.reference_point_y = 240
        self.last_lane_center = None
        self.last_boxes = []
        self.last_scores = []
        self.last_exec_ms = 0.0
        self._warp_cache = {}              # (h, w, mode) -> (M, out_size)

    def roi_rectangle_below(self, img, cutting_idx):
        return img[cutting_idx:, :]

    def warpping(self, image, srcmat, dstmat):
        h, w = image.shape[0], image.shape[1]
        M = cv2.getPerspectiveTransform(srcmat, dstmat)
        minv = cv2.getPerspectiveTransform(dstmat, srcmat)
        warped = cv2.warpPerspective(image, M, (w, h))
        return warped, minv

    def bird_convert(self, img, srcmat, dstmat):
        srcmat = np.float32(srcmat)
        dstmat = np.float32(dstmat)
        warped, _ = self.warpping(img, srcmat, dstmat)
        return warped

    def calculate_angle(self, x1, y1, x2, y2):
        # 수직선(직진 방향) 기준 각도. 0 = 직진, 음수 = 왼쪽, 양수 = 오른쪽. 전 구간 연속.
        return math.degrees(math.atan2(x2 - x1, y1 - y2))

    def is_lane_colored(self, img_bgr, box):
        if not USE_COLOR_FILTER:
            return True
        y1, x1, y2, x2 = [int(v) for v in box]
        h, w = img_bgr.shape[:2]
        y1, y2 = max(0, y1), min(h, y2)
        x1, x2 = max(0, x1), min(w, x2)
        if y2 <= y1 or x2 <= x1:
            return True
        crop = img_bgr[y1:y2, x1:x2]
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        mean_saturation = float(hsv[:, :, 1].mean())
        return mean_saturation <= LANE_MAX_SATURATION

    def detect_lane_center_x(self, xyxy_results, img_bgr=None):
        valid_boxes = []
        for box in xyxy_results:
            y1, x1, y2, x2 = box
            if img_bgr is not None and not self.is_lane_colored(img_bgr, box):
                continue
            box_center = (int(x1) + int(x2)) // 2
            valid_boxes.append((box, box_center))

        if not valid_boxes:
            return None

        # 이전 차선 위치 기반 추적 (꽃/잔디 오검출 배제)
        if self.last_lane_center is not None:
            MAX_SHIFT = 25
            closest_center = None
            min_dist = float('inf')
            for box, center in valid_boxes:
                dist = abs(center - self.last_lane_center)
                if dist <= MAX_SHIFT and dist < min_dist:
                    min_dist = dist
                    closest_center = center
            if closest_center is not None:
                return closest_center
            else:
                return None

        # 초기 탐색 모드: 가장 오른쪽 박스를 차선으로
        rightmost = -float('inf')
        best_center = None
        for box, center in valid_boxes:
            y1, x1, y2, x2 = box
            if x1 > rightmost:
                rightmost = x1
                best_center = center
        return best_center

    def compute_bev_geometry(self, img):
        """캡처 프레임을 비율 유지한 채 BEV_WORK_HEIGHT로 축소 후 SRC_RATIO/CUTTING_RATIO를 환산."""
        h, w = img.shape[0], img.shape[1]
        work_w = round(w * BEV_WORK_HEIGHT / h)
        work_img = cv2.resize(img, (work_w, BEV_WORK_HEIGHT))
        wh, ww = work_img.shape[0], work_img.shape[1]
        src_mat = [[round(rx * ww), round(ry * wh)] for rx, ry in SRC_RATIO]
        dst_mat = [[round(ww * 0.3), 0], [round(ww * 0.7), 0],
                   [round(ww * 0.7), wh], [round(ww * 0.3), wh]]
        cutting_idx = round(wh * CUTTING_RATIO)
        return work_img, src_mat, dst_mat, cutting_idx

    # ---------------- v13: 합성 호모그래피 ----------------
    def _build_warp(self, src_h, src_w, mode):
        """(원본 or work) -> 256x256 을 한 번에 보내는 3x3 행렬을 만든다.

        legacy 경로가 하던 일:
            원본 --S--> work(ww x wh) --B--> BEV(ww x wh) --crop(y>=cut)--> --R--> 256x256
        이 네 단계는 전부 사영변환이므로 M = R . C . B . S 로 합성 가능하다.
        """
        work_h = BEV_WORK_HEIGHT
        work_w = int(round(src_w * work_h / float(src_h)))

        src_mat = np.float32([[round(rx * work_w), round(ry * work_h)] for rx, ry in SRC_RATIO])
        dst_mat = np.float32([[round(work_w * 0.3), 0], [round(work_w * 0.7), 0],
                              [round(work_w * 0.7), work_h], [round(work_w * 0.3), work_h]])
        cut = int(round(work_h * CUTTING_RATIO))

        M_bev = cv2.getPerspectiveTransform(src_mat, dst_mat)          # work -> BEV

        # BEV -> (하단 crop) -> 256x256 : x' = x*256/ww,  y' = (y-cut)*256/(wh-cut)
        sx = 256.0 / work_w
        sy = 256.0 / (work_h - cut)
        M_roi = np.float32([[sx, 0.0, 0.0],
                            [0.0, sy, -sy * cut],
                            [0.0, 0.0, 1.0]])

        M = M_roi.dot(M_bev)
        if mode == "fastest":
            # 원본 -> work 축소까지 행렬에 흡수 (cv2.resize 자체를 없앰).
            # 주의: 1920 -> 256을 bilinear로 직접 샘플링하므로 에일리어싱이 생길 수 있다.
            #       검출률이 떨어지면 "fast"로 되돌릴 것.
            M_scale = np.float32([[work_w / float(src_w), 0.0, 0.0],
                                  [0.0, work_h / float(src_h), 0.0],
                                  [0.0, 0.0, 1.0]])
            M = M.dot(M_scale)
        return M.astype(np.float32), (work_w, work_h)

    def to_square_bev(self, img, timer=None):
        """프레임 -> 256x256 BEV. BEV_MODE에 따라 경로가 갈린다."""
        h, w = img.shape[0], img.shape[1]
        mode = BEV_MODE

        if mode == "legacy":
            with (timer.stage("bev_resize") if timer else _NullScope()):
                work_img, src_mat, dst_mat, cutting_idx = self.compute_bev_geometry(img)
            with (timer.stage("bev_warp") if timer else _NullScope()):
                bird_img = self.bird_convert(work_img, src_mat, dst_mat)
            with (timer.stage("bev_crop_resize") if timer else _NullScope()):
                roi_image = self.roi_rectangle_below(bird_img, cutting_idx=cutting_idx)
                square = cv2.resize(roi_image, (256, 256))
            return square

        key = (h, w, mode)
        if key not in self._warp_cache:
            self._warp_cache[key] = self._build_warp(h, w, mode)
        M, (work_w, work_h) = self._warp_cache[key]

        if mode == "fastest":
            with (timer.stage("bev_warp") if timer else _NullScope()):
                return cv2.warpPerspective(img, M, (256, 256), flags=cv2.INTER_LINEAR)

        # mode == "fast": 축소는 유지하고(에일리어싱 방지), 워프만 256으로 직접.
        # 보간은 반드시 INTER_LINEAR — legacy가 쓰던 기본값이고, INTER_AREA는 화질이 조금 나은 대신
        # 실측 22배 느리다(0.12ms -> 2.74ms). 여기서 AREA를 쓰면 워프에서 아낀 걸 전부 토해낸다.
        with (timer.stage("bev_resize") if timer else _NullScope()):
            work_img = cv2.resize(img, (work_w, work_h), interpolation=cv2.INTER_LINEAR)
        with (timer.stage("bev_warp") if timer else _NullScope()):
            return cv2.warpPerspective(work_img, M, (256, 256), flags=cv2.INTER_LINEAR)

    def process_frame(self, img, timer=None):
        # (1)(2) BEV + ROI + 256 리사이즈 (v13: 한 번의 워프로 합성 가능)
        resized_img = self.to_square_bev(img, timer=timer)

        # (3) 전처리 + DPU 추론
        with (timer.stage("preproc") if timer else _NullScope()):
            image_size = resized_img.shape[:2]
            image_data = np.array(pre_process(resized_img, (256, 256)), dtype=np.float32)

        with (timer.stage("dpu") if timer else _NullScope()):
            start_time = time.time()
            image[0, ...] = image_data.reshape(shapeIn[1:])
            job_id = dpu.execute_async(input_data, output_data)
            dpu.wait(job_id)
            end_time = time.time()
            self.last_exec_ms = (end_time - start_time) * 1000.0

        with (timer.stage("postproc") if timer else _NullScope()):
            conv_out0 = np.reshape(output_data[0], shapeOut0)
            conv_out1 = np.reshape(output_data[1], shapeOut1)
            yolo_outputs = [conv_out0, conv_out1]
            boxes, scores, classes = evaluate(yolo_outputs, image_size, class_names, anchors, YOLO_THRESHOLD)

            self.last_boxes = [[round(float(v), 1) for v in b] for b in boxes]
            self.last_scores = [round(float(s), 3) for s in scores]

            for box in boxes:
                cv2.rectangle(resized_img, (int(box[1]), int(box[0])),
                              (int(box[3]), int(box[2])), (0, 255, 0), 2)

            # (4) 차선 중심 + 각도
            center = self.detect_lane_center_x(boxes, img_bgr=resized_img)
            self.last_lane_center = center
            if center is None:
                return 0.0, resized_img

            angle = self.calculate_angle(self.reference_point_x, self.reference_point_y,
                                         center, self.point_detection_height)

            cv2.line(resized_img, (int(self.reference_point_x), self.reference_point_y),
                     (center, self.point_detection_height), (0, 0, 255), 3)
            cv2.circle(resized_img, (center, self.point_detection_height), 6, (0, 255, 255), -1)

        return angle, resized_img

    def debug_calibration(self, img):
        """src_mat 사다리꼴을 원본에 그리고, BEV/정사각형 결과도 함께 반환."""
        work_img, src_mat, dst_mat, cutting_idx = self.compute_bev_geometry(img)
        marked = work_img.copy()
        pts = np.array(src_mat, dtype=np.int32)
        cv2.polylines(marked, [pts], isClosed=True, color=(0, 255, 255), thickness=2)
        for i, (x, y) in enumerate(src_mat):
            cv2.circle(marked, (x, y), 5, (0, 0, 255), -1)
            cv2.putText(marked, str(i + 1), (x + 6, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        bird_img = self.bird_convert(work_img, src_mat, dst_mat)
        roi_image = self.roi_rectangle_below(bird_img, cutting_idx=cutting_idx)
        square = cv2.resize(roi_image, (256, 256))
        return marked, bird_img, square

    def compare_bev_modes(self, img):
        """BEV_MODE별 결과가 실제로 같은지 눈으로 확인 + 소요시간 비교.

        'fast'는 legacy와 화질이 같아야 하고, 'fastest'는 약간 뭉개질 수 있다.
        차선 검출률이 떨어지면 fastest를 포기하고 fast를 쓸 것.
        """
        global BEV_MODE
        saved = BEV_MODE
        out = {}
        try:
            for m in ("legacy", "fast", "fastest"):
                BEV_MODE = m
                self._warp_cache = {}
                self.to_square_bev(img)                     # 캐시 워밍업
                t0 = time.time()
                for _ in range(10):
                    sq = self.to_square_bev(img)
                out[m] = (sq, (time.time() - t0) * 100.0)   # ms/회
        finally:
            BEV_MODE = saved
            self._warp_cache = {}
        for m, (sq, ms) in out.items():
            print("  %-8s %6.1f ms/frame" % (m, ms))
        base = out["legacy"][0].astype(np.int16)
        for m in ("fast", "fastest"):
            diff = np.abs(out[m][0].astype(np.int16) - base)
            print("  legacy 대비 %-8s 평균 화소차 %.2f / 최대 %d" % (m, diff.mean(), diff.max()))
        return out


print("ImageProcessor(v13: BEV_MODE=%s, 합성 호모그래피 + 단계별 계측) ready" % BEV_MODE)


## BEV 모드 벤치마크 (주행 전 1회)

`fast` 는 `legacy` 와 화소가 거의 같아야 합니다(반올림 차이만). `fastest` 는 1920→256 직접
샘플링이라 뭉개질 수 있으니, 화소차가 크면 `fast` 를 쓰세요. 속도만 보고 고르지 말 것.

In [ ]:
# 카메라 한 장 잡아서 세 모드 비교 (속도 + legacy 대비 화소차)
_cap = cv2.VideoCapture(0)
_cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAPTURE_W)
_cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAPTURE_H)
_ok, _f = _cap.read()
_cap.release()
if not _ok:
    print("카메라 캡처 실패")
else:
    print("입력 프레임:", _f.shape)
    _ip = ImageProcessor()
    _out = _ip.compare_bev_modes(_f)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, m in zip(axes, ("legacy", "fast", "fastest")):
        ax.imshow(cv2.cvtColor(_out[m][0], cv2.COLOR_BGR2RGB)); ax.set_title(m); ax.axis("off")
    plt.tight_layout(); plt.show()


## 차선 밟음 라벨링 (q / w / a / s)

주행 중 차를 따라가며 **차선을 밟고 있는 바퀴의 키를 누르고 있으면** 그 구간이 로그에 남습니다.

| 키 | 바퀴 |
|---|---|
| `q` | 왼쪽 앞 (LF) |
| `w` | 오른쪽 앞 (RF) |
| `a` | 왼쪽 뒤 (LR) |
| `s` | 오른쪽 뒤 (RR) |

**이 셀을 Driving! 셀보다 먼저 실행하세요.** 현재 눌린 상태가 위젯에 실시간으로 표시됩니다.

> 기존 수동주행 셀의 키보드 로직은 `Jupyter.notebook.kernel` 을 쓰는데 이건 classic Notebook
> 전용이라 지금 쓰시는 **JupyterLab에서는 동작하지 않습니다**(콘솔에 에러만 찍고 조용히 무시됨).
> 그래서 여기서는 위젯 comm을 통한 Lab 호환 방식으로 다시 만들었습니다.
> `q`/`s` 가 수동주행의 정지/후진과 겹치므로, 이 셀이 그 핸들러를 먼저 해제합니다.

In [ ]:
# ===== 차선 밟음 라벨링 (q / w / a / s) =====
# 주행 중 사람이 차를 따라가며, 차선을 밟고 있는 바퀴의 키를 '누르고 있는' 동안 기록한다.
#   q = 왼쪽 앞(LF)   w = 오른쪽 앞(RF)   a = 왼쪽 뒤(LR)   s = 오른쪽 뒤(RR)
# 이 셀을 먼저 실행한 뒤 아래 Driving! 셀을 실행할 것. 라벨은 텔레메트리 "wheel" 필드에 들어간다.
#
# ⚠️ 기존 '수동 주행' 셀의 키보드 로직은 여기서 쓸 수 없다.
#    그 코드는 Jupyter.notebook.kernel 을 쓰는데 이건 classic Notebook 전용이고,
#    지금 쓰는 환경은 JupyterLab(/lab/tree/...)이라 Jupyter.notebook 이 undefined다.
#    (그 셀도 이 경우 콘솔에 에러만 찍고 조용히 아무것도 안 한다.)
#    그래서 여기서는 ipywidgets 위젯의 DOM input에 값을 써 넣고 'input' 이벤트를 발생시켜
#    위젯 comm으로 커널까지 동기화하는 방식을 쓴다. Lab / classic 양쪽에서 동작한다.
#
# ⚠️ q와 s는 수동주행 셀에서 각각 '정지'와 '후진'에 바인딩돼 있다.
#    아래 JS가 그 핸들러를 먼저 제거하므로, 수동주행을 다시 쓰려면 그 셀을 재실행할 것.

from IPython.display import Javascript, display
import ipywidgets as widgets

WHEEL_ORDER = ("LF", "RF", "LR", "RR")
WHEEL_KEYS = {"q": "LF", "w": "RF", "a": "LR", "s": "RR"}
WHEEL_LABEL_ENABLE = True     # False면 라벨링 끔(주행에는 영향 없음)

try:
    import keyboard as os_keyboard   # 보드에 USB 키보드를 직접 꽂은 경우
except ImportError:
    os_keyboard = None


class WheelLabeler:
    """차선을 밟은 바퀴를 사람이 실시간으로 라벨링한다.

    입력 경로 두 가지를 OR로 합친다:
      1) 브라우저 키보드 -> 위젯 comm -> 커널 (원격 Jupyter에서 동작, 창 포커스 필요)
      2) 보드에 직접 꽂은 USB 키보드 (keyboard 모듈, 차를 따라 걸어갈 때 유리)
    """

    def __init__(self):
        self.probe = widgets.Text(value="0000", description="", disabled=True)
        self.probe.add_class("wheel-key-probe")
        self.probe.layout.width = "90px"
        self.status = widgets.HTML(value=self._render([]))
        self.box = widgets.HBox([self.status, self.probe])
        self.counts = {w: 0 for w in WHEEL_ORDER}
        self.frames = 0
        self._os_ok = os_keyboard is not None
        self._os_warned = False

    @staticmethod
    def _render(touched):
        cells = []
        for w in WHEEL_ORDER:
            on = w in touched
            cells.append(
                "<span style='display:inline-block;min-width:44px;text-align:center;"
                "padding:4px 8px;margin-right:6px;border-radius:6px;font-weight:600;"
                "font-family:ui-monospace,monospace;background:%s;color:%s'>%s</span>"
                % ("#e34948" if on else "#eceae2", "#fff" if on else "#8f8e85", w)
            )
        key = "<span style='color:#8f8e85;font-size:12px'>q=LF w=RF a=LR s=RR</span>"
        return "<div style='display:flex;align-items:center;gap:12px'>" + "".join(cells) + key + "</div>"

    def install(self):
        display(self.box)
        display(Javascript(_WHEEL_JS))
        print("[LABEL] q=왼쪽앞 w=오른쪽앞 a=왼쪽뒤 s=오른쪽뒤 — 밟는 동안 누르고 계세요.")
        print("[LABEL] 브라우저 입력은 이 탭에 포커스가 있어야 잡힙니다."
              + ("  (보드 USB 키보드도 함께 감시)" if self._os_ok else "  (보드 USB 키보드 없음)"))

    def read(self):
        touched = []
        v = self.probe.value or ""
        if len(v) == len(WHEEL_ORDER):
            for ch, name in zip(v, WHEEL_ORDER):
                if ch == "1":
                    touched.append(name)
        if self._os_ok:
            try:
                for k, name in WHEEL_KEYS.items():
                    if os_keyboard.is_pressed(k) and name not in touched:
                        touched.append(name)
            except Exception as e:
                self._os_ok = False
                if not self._os_warned:
                    print("[LABEL] 보드 키보드 비활성화: %s" % e)
                    self._os_warned = True
        return touched

    def note(self, touched):
        """프레임당 1회. 집계 + 화면 표시 갱신."""
        self.frames += 1
        for w in touched:
            self.counts[w] += 1
        try:
            self.status.value = self._render(touched)
        except Exception:
            pass
        return touched

    def summary(self):
        if self.frames == 0:
            return "라벨 없음"
        return " ".join("%s %d(%.0f%%)" % (w, self.counts[w], 100.0 * self.counts[w] / self.frames)
                        for w in WHEEL_ORDER)


_WHEEL_JS = r"""
(function() {
    // 수동주행 셀의 기존 핸들러 제거 (q=정지, s=후진 과 충돌)
    if (window._manualDriveHandlers) {
        document.removeEventListener('keydown', window._manualDriveHandlers.down, true);
        document.removeEventListener('keyup', window._manualDriveHandlers.up, true);
        window.removeEventListener('blur', window._manualDriveHandlers.blur);
        window._manualDriveHandlers = null;
        console.log('[WHEEL LABEL] 수동주행 키 핸들러를 해제했습니다.');
    }
    // 이 셀 재실행 시 중복 바인딩 방지
    if (window._wheelLabelHandlers) {
        document.removeEventListener('keydown', window._wheelLabelHandlers.down, true);
        document.removeEventListener('keyup', window._wheelLabelHandlers.up, true);
        window.removeEventListener('blur', window._wheelLabelHandlers.blur);
    }

    const ORDER = ['q', 'w', 'a', 's'];        // LF, RF, LR, RR
    const state = { q: false, w: false, a: false, s: false };

    // Jupyter.notebook.kernel 을 쓰지 않는다 (JupyterLab에는 없음).
    // 대신 위젯의 DOM input 값을 바꾸고 'input' 이벤트를 던지면 ipywidgets가 커널로 동기화한다.
    function push() {
        const el = document.querySelector('.wheel-key-probe input');
        if (!el) return;
        const v = ORDER.map(k => state[k] ? '1' : '0').join('');
        if (el.value === v) return;
        // disabled 위젯이라도 값 설정 + 이벤트 디스패치는 동작한다.
        const setter = Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value').set;
        setter.call(el, v);
        el.dispatchEvent(new Event('input', { bubbles: true }));
        el.dispatchEvent(new Event('change', { bubbles: true }));
    }

    function onDown(e) {
        const k = e.key ? e.key.toLowerCase() : '';
        if (!ORDER.includes(k)) return;
        e.preventDefault(); e.stopPropagation();
        if (e.repeat) return;              // OS 자동반복 무시
        state[k] = true; push();
    }
    function onUp(e) {
        const k = e.key ? e.key.toLowerCase() : '';
        if (!ORDER.includes(k)) return;
        e.preventDefault(); e.stopPropagation();
        state[k] = false; push();
    }
    function onBlur() {
        // 포커스를 잃으면 전부 해제. 안 그러면 눌린 채로 굳어 라벨이 통째로 오염된다.
        ORDER.forEach(k => state[k] = false); push();
    }
    document.addEventListener('keydown', onDown, true);
    document.addEventListener('keyup', onUp, true);
    window.addEventListener('blur', onBlur);
    window._wheelLabelHandlers = { down: onDown, up: onUp, blur: onBlur };
    console.log('[WHEEL LABEL] q/w/a/s 바인딩 완료');
})();
"""

wheel_labeler = WheelLabeler()
if WHEEL_LABEL_ENABLE:
    wheel_labeler.install()
else:
    print("[LABEL] WHEEL_LABEL_ENABLE=False — 라벨링 꺼짐")


## Driving!

**첫 주행은 `RACING_LINE_ENABLE = False` 로 기준선부터 잡으세요.** 프레임레이트 개선만의
효과를 분리해서 봐야 라인 효과를 나중에 구분할 수 있습니다.

권장 순서:
1. `RACING_LINE_ENABLE=False`, `BEV_MODE="fast"` → 프레임레이트 효과 측정
2. `RACING_LINE_ENABLE=True`, `RACING_LINE_GAIN=0.5` → 라인 효과 측정
3. 차선 여유를 보며 `RACING_LINE_GAIN` 을 0.7 → 1.0 으로

In [9]:
# ===== 실행 (블랙박스 기록 모드) =====
# #  - 좌/우 바퀴가 같은 방향(전진)으로 도는지
#  - 차선이 화면 오른쪽이면 우조향인지 (반대면 위 셀의 STEER_DIR 을 -1 로)
# 주행이 끝나면(Ctrl+C 또는 셀 중지) 기록이 자동으로 닫히고 run_* 폴더 경로가 출력됨.
# 맥에서 가져와서 replay_viewer.html 로 분석:
#   scp -r xilinx@<보드IP>:/home/xilinx/jupyter_notebooks/KGIC/driveCode/logs/run_* ~/AIchip/logs/
controller = RobotController()
controller.run(camera_index=0)

모터 정지 및 종료.


## 수동 주행 (WASD)

In [12]:
# ===== 수동 주행 (WASD) =====
# 목적: DPU/비전 없이 수동으로 모터/조향 폐루프를 조작.
# 조작: W/S 누르고 있는 동안 전/후진 유지, 떼는 즉시 정지.
#       A/D를 누르고 있는 동안 조향 목표가 초당 MANUAL_STEER_RATE만큼 누적 변경됨.
#       A/D를 떼어도 현재 조향각을 유지. P를 누르면 전체 자동 주차 순서(차선추종->초음파->주차)를 시작. Q 또는 아래 '정지' 버튼으로 종료.
# 구조: 브라우저(JS)가 keydown/keyup을 capture 단계에서 가로채 커널의 전역 dict(_manual_keys)를 갱신하고,
#       파이썬 백그라운드 쓰레드가 그 값을 MANUAL_HZ로 읽어 RobotController.control_motors()를 그대로 재사용.
#       data_collection.ipynb의 `keyboard` 모듈(OS 레벨 후킹, 로컬 물리 키보드 필요) 대신 브라우저 이벤트를 쓰므로
#       원격 Jupyter에서도 항상 동작하고, 키를 떼는 순간 즉시 반영됨(누적/리셋키 불필요).
#       Driving! 셀(controller.run())이 실행 중이면 모터를 두 곳에서 동시에 건드리게 되니
#       먼저 Ctrl+C로 그 셀을 중지한 뒤 이 셀을 실행하세요.

import threading
try:
    import keyboard as os_keyboard  # USB wireless keyboard connected directly to the PYNQ board.
except ImportError:
    os_keyboard = None
from IPython.display import Javascript, display
import ipywidgets as widgets

MANUAL_DRIVE_SPEED = 90     # W/S 고정 구동 속도 (0~100)
MANUAL_STEER_TARGET = 20    # A/D 고정 조향 목표 (풀 좌/우, ±20)
MANUAL_HZ = 100              # 폴링/제어 주기(Hz) — control_loop과 동일

MANUAL_STEER_RATE = 30.0   # A/D steering-target ramp rate (degrees/s)

_manual_keys = {'w': False, 'a': False, 's': False, 'd': False, 'p': False, 'q': False}
_manual_stop = False

_manual_js = r"""
(function() {
    // 셀 재실행 시 이전 리스너 정리 (중복 바인딩 방지)
    if (window._manualDriveHandlers) {
        document.removeEventListener('keydown', window._manualDriveHandlers.down, true);
        document.removeEventListener('keyup', window._manualDriveHandlers.up, true);
        window.removeEventListener('blur', window._manualDriveHandlers.blur);
    }
    if (typeof Jupyter === 'undefined' || !Jupyter.notebook) {
        console.error('[MANUAL DRIVE] Jupyter.notebook을 찾을 수 없음 (classic Notebook 필요, JupyterLab 콘솔 등에서는 미지원).');
        return;
    }
    const kernel = Jupyter.notebook.kernel;
    const watched = ['w', 'a', 's', 'd', 'p', 'q'];

    function setKey(key, state) {
        if (key === 'q') {
            if (state) kernel.execute('_manual_stop = True; manual_ctrl.stop_flag = True');
            return;
        }
        kernel.execute(`_manual_keys['${key}'] = ${state ? 'True' : 'False'}`);
    }
    // capture 단계(true)에서 가로채서 preventDefault -> 코드 셀에 문자로 입력되거나
    // Jupyter 커맨드모드 단축키(a=위에 셀 삽입 등)로 오작동하는 것을 막음.
    function onDown(e) {
        const k = e.key.toLowerCase();
        if (!watched.includes(k)) return;
        e.preventDefault(); e.stopPropagation();
        if (!e.repeat) setKey(k, true);   // OS 자동반복 무시, 최초 keydown에서만 상태 전환
    }
    function onUp(e) {
        const k = e.key.toLowerCase();
        if (!watched.includes(k)) return;
        e.preventDefault(); e.stopPropagation();
        setKey(k, false);
    }
    function onBlur() {
        // 창 포커스를 잃으면 눌림 상태 전부 해제 (모터가 켜진 채로 고정되는 것 방지)
        ['w', 'a', 's', 'd', 'p'].forEach(k => kernel.execute(`_manual_keys['${k}'] = False`));
    }
    document.addEventListener('keydown', onDown, true);
    document.addEventListener('keyup', onUp, true);
    window.addEventListener('blur', onBlur);
    window._manualDriveHandlers = { down: onDown, up: onUp, blur: onBlur };
    console.log('[MANUAL DRIVE] WASD/P 바인딩 완료 (P = 전체 주차 시작, Q = 정지)');
})();
"""
display(Javascript(_manual_js))


class ManualInput:
    def __init__(self):
        self.hardware_enabled = os_keyboard is not None
        self.hardware_error_reported = False

    def read_keys(self):
        # Browser keys support the computer keyboard; OS keys support a USB keyboard on the PYNQ board.
        keys = dict(_manual_keys)
        if not self.hardware_enabled:
            return keys
        try:
            for key in keys:
                keys[key] = keys[key] or os_keyboard.is_pressed(key)
            return keys
        except Exception as error:
            self.hardware_enabled = False
            if not self.hardware_error_reported:
                print(f'[MANUAL DRIVE] board keyboard disabled: {error}')
                self.hardware_error_reported = True
            return keys


class ManualDriver:
    def __init__(self, ctrl):
        self.ctrl = ctrl
        self.stop_flag = False
        self.input = ManualInput()
        self.steering_target = 0.0
        self._p_armed = True
        self._parking_run_active = threading.Event()

    def _watch_parking_stop(self):
        global _manual_stop
        while self._parking_run_active.is_set():
            keys = self.input.read_keys()
            if keys.get('q', False) or _manual_stop:
                self.ctrl.stop_flag = True
                self.stop_flag = True
                _manual_stop = True
                print('[MANUAL DRIVE] Q pressed: parking attempt stopped.')
                return
            time.sleep(0.03)

    def loop(self):
        global _manual_stop
        period = 1.0 / MANUAL_HZ
        while not self.stop_flag and not _manual_stop:
            keys = self.input.read_keys()
            w, a, s, d = keys['w'], keys['a'], keys['s'], keys['d']
            if keys.get('q', False):
                _manual_stop = True
                self.ctrl.stop_flag = True
                break

            # P hands off one full attempt, then this same loop resumes manual driving.
            if not keys['p']:
                self._p_armed = True
            if keys['p'] and self._p_armed:
                self._p_armed = False
                self.ctrl.arm_parking_sequence('manual_p')
                print('[MANUAL DRIVE] P pressed: full parking sequence started.')
                self._parking_run_active.set()
                stop_watcher = threading.Thread(target=self._watch_parking_stop, daemon=True)
                stop_watcher.start()
                self.ctrl.run(camera_index=0, return_to_manual=True)
                self._parking_run_active.clear()
                self.steering_target = 0.0
                if self.stop_flag or _manual_stop:
                    break
                continue

            if w and not s:
                speed = MANUAL_DRIVE_SPEED
            elif s and not w:
                speed = -MANUAL_DRIVE_SPEED
            else:
                speed = 0
            self.ctrl.left_speed = speed
            self.ctrl.right_speed = speed

            # A/D ramp the target while held. Releasing either key preserves the current wheel angle.
            if a and not d:
                self.steering_target -= MANUAL_STEER_RATE * period
            elif d and not a:
                self.steering_target += MANUAL_STEER_RATE * period
            self.steering_target = float(np.clip(self.steering_target, -20.0, 20.0))
            self.ctrl.steering_angle = self.steering_target

            self.ctrl.control_motors()   # Drive write + steering feedback control step.
            time.sleep(period)

        # 종료: 완전 정지
        self.ctrl.left_speed = 0
        self.ctrl.right_speed = 0
        self.ctrl.steering_angle = 0.0
        self.ctrl.control_motors()
        self.ctrl.stay()
        print("[MANUAL DRIVE] 정지 및 종료.")


manual_ctrl = RobotController()
# Manual input shares the PD motor controller, but must never start autonomous parking.
manual_ctrl.parking_enabled = False
manual_driver = ManualDriver(manual_ctrl)
manual_thread = threading.Thread(target=manual_driver.loop, daemon=True)
manual_thread.start()

_stop_btn = widgets.Button(description="정지 (또는 Q)", button_style="danger")
def _stop_manual(_=None):
    global _manual_stop
    _manual_stop = True
    manual_driver.stop_flag = True
    manual_ctrl.stop_flag = True
_stop_btn.on_click(_stop_manual)
display(_stop_btn)

print("[MANUAL DRIVE] 시작됨 — W/A/S/D로 조종, P로 전체 자동 주차를 시작.")
print("[MANUAL DRIVE] Q 또는 위 정지 버튼으로 종료.")

<IPython.core.display.Javascript object>

/usr/local/share/pynq-venv/lib/python3.10/site-packages/keyboard/_nixkeyboard.py:110: UserWarning: Failed to create a device file using `uinput` module. Sending of events may be limited or unavailable depending on plugged-in devices.
  device = aggregate_devices('kbd')


Button(button_style='danger', description='정지 (또는 Q)', style=ButtonStyle())

[MANUAL DRIVE] 시작됨 — W/A/S/D로 조종, P로 전체 자동 주차를 시작.
[MANUAL DRIVE] Q 또는 위 정지 버튼으로 종료.


In [10]:
import spidev
import time
import sys

# SPI 설정
spi0 = spidev.SpiDev()

spi0.open(0, 0) 

# SPI 속도 설정 (최대 20MHz, Pmod AD1은 최대 1MHz 권장)
spi0.max_speed_hz = 20000000

# SPI 모드 설정 (모드 0)
spi0.mode = 0b00

def read_adc(spi):
    adc_response = spi.xfer2([0x00, 0x00])
    adc_value = ((adc_response[0] & 0x0F) << 8) | adc_response[1]
    return adc_value 

try:
    while True:
        # D0 채널에서 데이터 읽기
        adc_value0 = read_adc(spi0)
        # 출력 내용을 덮어쓰기
        #sys.stdout.write(f"\rD0 채널 값: {adc_value0*3.3/4096}")
        sys.stdout.write(f"\rD0 채널 값: {adc_value0}")
        sys.stdout.flush()
        time.sleep(0.5)
except KeyboardInterrupt:
    spi0.close()


#우측 1294
#좌측 1883
#직진 1530

D0 채널 값: 1314